In [2]:
!pip install google-cloud-bigquery pandas scikit-learn xgboost matplotlib -q

In [3]:
from google.colab import auth
auth.authenticate_user()
print("Authenticated")

Authenticated


In [4]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "spotify-churn-pipeline"

client = bigquery.Client(project=PROJECT_ID)

query = """
SELECT *
FROM `spotify-churn-pipeline.sparkify_churn.user_features`
"""

df = client.query(query).to_dataframe()
print(df.shape)
df.head()

(225, 6)


,userId,total_listening_hours,average_skip_rate,active_days_in_last_30,total_events,has_churned
0,19,15.133593,0.009259,0,259,0
1,44,29.545858,0.004662,0,512,0
2,63,6.431067,0.034483,0,107,0
3,133,2.230323,0.000000,0,44,0
4,135,0.443457,0.000000,0,6,0


In [5]:
# Class balance check
print(df['has_churned'].value_counts())
print(df['has_churned'].value_counts(normalize=True))

has_churned
0    173
1     52
Name: count, dtype: Int64
has_churned
0    0.768889
1    0.231111
Name: proportion, dtype: Float64


In [6]:
# Summary stats for the features
df.describe()

,userId,total_listening_hours,average_skip_rate,active_days_in_last_30,total_events,has_churned
count,225.0,225.000000,225.000000,225.0,225.0,225.0
mean,65391.013333,70.155089,0.011895,5.635556,1236.24,0.231111
std,105396.477919,76.499001,0.010759,5.606961,1329.531716,0.422483
min,2.0,0.192130,0.000000,0.0,6.0,0.0
25%,60.0,16.009215,0.006974,1.0,296.0,0.0
50%,116.0,46.692719,0.009479,4.0,848.0,0.0
75%,100017.0,109.000415,0.013242,9.0,1863.0,0.0
max,300025.0,553.098588,0.090909,26.0,9632.0,1.0


In [7]:
from sklearn.model_selection import train_test_split

# Drop userId (identifier, not a real feature) and separate features from target
X = df.drop(columns=['userId', 'has_churned'])
y = df['has_churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # preserves the 23%/77% churn ratio in both splits
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.3f}")
print(f"Test churn rate: {y_test.mean():.3f}")

Train shape: (180, 4), Test shape: (45, 4)
Train churn rate: 0.233
Test churn rate: 0.222


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Train the model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
    class_weight='balanced'  # helps account for the 23%/77% imbalance
)

rf_model.fit(X_train, y_train)

# Predict on the test set
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]  # probability of churn (class 1)

# Evaluate
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.3f}")

Classification Report:
              precision    recall  f1-score   support

         0.0       0.89      0.97      0.93        35
         1.0       0.86      0.60      0.71        10

    accuracy                           0.89        45
   macro avg       0.88      0.79      0.82        45
weighted avg       0.89      0.89      0.88        45

Confusion Matrix:
[[34  1]
 [ 4  6]]
ROC-AUC Score: 0.911


In [9]:
import pandas as pd

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance)

                  feature  importance
2  active_days_in_last_30    0.513100
1       average_skip_rate    0.217250
0   total_listening_hours    0.140193
3            total_events    0.129457


In [10]:
# Try a lower threshold to catch more churners (trade precision for recall)
threshold = 0.35
y_pred_adjusted = (y_pred_proba >= threshold).astype(int)

print(f"Threshold: {threshold}")
print(classification_report(y_test, y_pred_adjusted))
print(confusion_matrix(y_test, y_pred_adjusted))

Threshold: 0.35
              precision    recall  f1-score   support

         0.0       0.92      0.94      0.93        35
         1.0       0.78      0.70      0.74        10

    accuracy                           0.89        45
   macro avg       0.85      0.82      0.83        45
weighted avg       0.89      0.89      0.89        45

[[33  2]
 [ 3  7]]


In [11]:
# Generate predictions for ALL users (not just test set) using the trained model
X_all = df.drop(columns=['userId', 'has_churned'])
all_probabilities = rf_model.predict_proba(X_all)[:, 1]

# Build the output DataFrame
predictions_df = pd.DataFrame({
    'user_id': df['userId'],
    'churn_probability': all_probabilities
})

# Segment into risk buckets
def assign_risk_segment(prob):
    if prob >= 0.6:
        return 'High Risk'
    elif prob >= 0.35:
        return 'Medium Risk'
    else:
        return 'Low Risk'

predictions_df['risk_segment'] = predictions_df['churn_probability'].apply(assign_risk_segment)

print(predictions_df['risk_segment'].value_counts())
predictions_df.head(10)

risk_segment
Low Risk       165
High Risk       39
Medium Risk     21
Name: count, dtype: int64


,user_id,churn_probability,risk_segment
0,19,0.417755,Medium Risk
1,44,0.853426,High Risk
2,63,0.583605,Medium Risk
3,133,0.521935,Medium Risk
4,135,0.248169,Low Risk
5,149,0.387915,Medium Risk
6,200010,0.459475,Medium Risk
7,300003,0.490618,Medium Risk
8,300024,0.572925,Medium Risk
9,34,0.021558,Low Risk


In [12]:
from google.cloud import bigquery

client = bigquery.Client(project="spotify-churn-pipeline")

table_id = "spotify-churn-pipeline.sparkify_churn.churn_predictions"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # overwrite if this table already exists
)

job = client.load_table_from_dataframe(predictions_df, table_id, job_config=job_config)
job.result()  # wait for the job to finish

print(f"Loaded {job.output_rows} rows into {table_id}")

Loaded 225 rows into spotify-churn-pipeline.sparkify_churn.churn_predictions
